# 3. Hybrid dynamic-GEM state-space model

## Goal

Does the state-space approach remain useful when metabolic trajectories arise from a dynamic Yeast9/GEM constraint system rather than the simple controlled ODE?

This notebook is a readable research record. It follows the actual hand-offs in order and loads the saved evidence by default; it does **not** hide the experiment behind a one-cell runner.


## Pipeline at a glance

```text
goal → declared generator → observable/lockbox split → model setup & training
     → candidate or condition screen → matched comparison → interpretation
```

Each section below corresponds to one of these hand-offs.


In [ ]:
# Run this notebook from the repository root.
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

REGENERATE = False  # Cached artifacts are the default; no expensive solve runs implicitly.

def artifact(relative_path: str) -> Path:
    """Fail with a useful message rather than silently replacing evidence."""
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f"Missing cached artifact: {path}")
    return path

def show(frame, n=8):
    # `print` keeps this notebook usable in a plain Python kernel as well as Jupyter.
    print(frame.head(n).to_string(index=False))
    print(f"{len(frame):,} rows × {len(frame.columns):,} columns")


## 1. Experimental contract

The experiment has a declared observation boundary. “Observable” means the learner may use it; “lockbox” means it may be generated and audited but must not be used as a deployable feature.


In [ ]:
from yeast_validation import run_dfba_state_machine_validation as dfba
from yeast_validation import gem_gsm_surrogate as surrogate

cfg = dfba.DFBAConfig()
print("GSM control head:", surrogate.CONTROL_COLUMNS)
print("GSM/surrogate flux outputs:", surrogate.SURROGATE_OUTPUT_COLUMNS)
print("observable: environment, biomass, product, selected reporters")
print("lockbox: reaction bounds, fluxes, burden states, enzyme capacities")


## 2. Data generator

At every interval, the generator applies environmental and dynamic metabolic constraints, solves staged LP/pFBA, then integrates biomass and β-carotene forward. Fluxes, bounds, and causal metabolic state are lockboxed.

The next cell exposes the generator’s first concrete hand-off. It is deliberately small/inspection-only where generating the full campaign is expensive.


In [ ]:
# The data generator is intentionally visible: it returns each hand-off table.
dataset = dfba.generate_dataset(cfg.dataset_seeds[0], cfg, fast=True)
traj, reporters, fluxes, manifest = dataset[:4]
show(manifest[[c for c in ["culture_id", "split_type", "temperature", "pH", "DO"] if c in manifest]])
show(traj)


## 3. Model setup and training contract

The hybrid learner consumes only environment plus observable biomass/product/reporter history. Its control head produces six compact metabolic controls; the frozen GSM surrogate maps controls and environment to flux predictions during training.

Training is not automatically started in this notebook. The cached training/evaluation artifacts below are the evidence record; regeneration must be an intentional, parameterized action.


In [ ]:
# Make the experiment hand-off inspectable before looking at aggregate metrics.
for name, relative_path in [('metrics', 'results/experiment_3b_state_machine_dfba/summary_metrics.csv')]:
    path = artifact(relative_path)
    print(f"{name}: {path.relative_to(ROOT)}")


## 4. Screening / selection stage

This phase screens model variants on held-out whole cultures, with the same observable interface for every deployable comparison.

The screen is intentionally shown separately from final verification, so a virtual score cannot be mistaken for an exact outcome.


In [ ]:
# Load the primary evidence table and inspect its schema before aggregation.
metrics = pd.read_csv(artifact('results/experiment_3b_state_machine_dfba/summary_metrics.csv'))
show(metrics)


## 5. Matched comparison

Compare product and biomass error by experiment and model class, never by giving a competitor lockbox fluxes or bounds.


In [ ]:
# Aggregate only over fields that exist in this version of the cached record.
comparison = metrics.groupby(['model', 'split'], dropna=False).mean(numeric_only=True)
show(comparison.reset_index() if hasattr(comparison, "reset_index") else comparison)


## 6. Analysis view

The plot is intentionally generic: it exposes every numeric evidence column so the reader can select the metric relevant to the claim, rather than hard-coding an attractive subset.


In [ ]:
numeric = metrics.select_dtypes("number")
if numeric.shape[1]:
    ax = numeric.plot(kind="box", rot=45, figsize=(11, 4), title="Cached evidence: numeric metric distribution")
    ax.set_ylabel("recorded metric value")
    plt.tight_layout()
else:
    print("This artifact has no numeric columns to plot.")


## 7. Interpretation, scope, and next hand-off

This is the first phase where the ML-to-GSM hand-off is explicit: learned controls constrain a metabolic solve; they are not themselves biological ground truth.

### Reproduction boundary

The cells above reveal the inputs and artifacts without launching an expensive campaign. To regenerate, use the explicit command below only after reviewing its declared inputs and output destination.


In [ ]:
if REGENERATE:
    # This guard prevents accidental solver/campaign execution.
    dfba.run_all(fast=False, write=True, backend=None)
